# 1 · Build the Transformer, piece by piece

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajdama/paper-to-pytorch/blob/main/notebooks/01_build_the_transformer.ipynb)

This notebook walks through Steps 01–10 of [paper-to-pytorch](https://github.com/rajdama/paper-to-pytorch):
every component of *Attention Is All You Need*, each one tested against the paper's
formula or PyTorch's own implementation. Run the cells top to bottom.

In [ ]:
# Get the code (Colab already has PyTorch and matplotlib installed)
import os
if not os.path.exists("step03_attention"):
    if not os.path.exists("paper-to-pytorch"):
        !git clone -q https://github.com/rajdama/paper-to-pytorch.git
    %cd paper-to-pytorch

import torch
from IPython.display import Image, display
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine for this notebook)")

## Step 01–02 · Embeddings and positional encoding
Tokens become vectors, and every position gets a unique sine/cosine "signature" added on top.

In [ ]:
from step01_embeddings import TokenEmbedding
from step02_positional_encoding import PositionalEncoding

emb = TokenEmbedding(vocab_size=100, d_model=16)
pe = PositionalEncoding(d_model=16, dropout=0.0)
tokens = torch.tensor([[5, 42, 7, 7]])           # note: token 7 appears twice
x = pe(emb(tokens))
print("shape:", tuple(x.shape))
print("same token, different positions -> different vectors:", not torch.allclose(x[0, 2], x[0, 3]))

!python -m step02_positional_encoding.demo
display(Image("step02_positional_encoding/pe_heatmap.png"))

## Step 03 · Scaled dot-product attention
$\text{softmax}(QK^\top/\sqrt{d_k})\,V$. The demo uses hand-made vectors where animals look alike and colours look alike.

In [ ]:
!python -m step03_attention.demo
display(Image("step03_attention/attention_heatmap.png"))

**Try it:** why divide by $\sqrt{d_k}$? Make the vectors 10× bigger and compare the attention with and without scaling.

In [ ]:
import torch.nn.functional as F
q = torch.randn(1, 6, 64) * 10
k = torch.randn(1, 6, 64) * 10
for scaled in (True, False):
    scores = q @ k.transpose(-2, -1) / (64 ** 0.5 if scaled else 1)
    w = F.softmax(scores, dim=-1)
    print(f"scaled={scaled!s:5}  largest weight in each row:", w.max(-1).values.squeeze().numpy().round(3))
# Unscaled: rows become one-hot, softmax saturates, and gradients vanish.

## Step 04–05 · Masks and multi-head attention

In [ ]:
from step04_masking import make_causal_mask
from step05_multi_head_attention import MultiHeadAttention

print(make_causal_mask(5)[0, 0].int())
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 5, 64)
out = mha(x, x, x, make_causal_mask(5))
print("output:", tuple(out.shape), " attention weights per head:", tuple(mha.attn_weights.shape))

!python -m step05_multi_head_attention.demo
display(Image("step05_multi_head_attention/heads.png"))

## Step 06–09 · Feed-forward, Add & Norm, encoder, decoder

In [ ]:
from step08_encoder import Encoder
from step09_decoder import Decoder

enc = Encoder(num_layers=2, d_model=64, num_heads=8, d_ff=256)
dec = Decoder(num_layers=2, d_model=64, num_heads=8, d_ff=256)
src, tgt = torch.randn(1, 9, 64), torch.randn(1, 5, 64)
memory = enc(src)
print("encoder memory:", tuple(memory.shape))
print("decoder output:", tuple(dec(tgt, memory, tgt_mask=make_causal_mask(5)).shape),
      " <- 5 target tokens reading 9 source tokens through cross-attention")

## Step 10 · The full Transformer
The paper's base configuration: $d_{model}=512$, 8 heads, 6 layers, $d_{ff}=2048$.

In [ ]:
!python -m step10_full_transformer.summary

## Run every test in the repo

In [ ]:
!python -m pytest -q

**Next:** [2 · Train the Transformer](https://colab.research.google.com/github/rajdama/paper-to-pytorch/blob/main/notebooks/02_train_the_transformer.ipynb)